GRM / Genotype Preparation
Prepare a high-quality, pruned genotype set for Step-1 ridge regression.

This ensures that only common, well-QC’d variants with consistent IDs are retained,
providing REGENIE with a clean, reliable input for building genome-wide polygenic predictors.

In [3]:
# Import required libraries
import sys
import os 
import numpy as np
import pandas as pd
from datetime import datetime

# Ensuring dsub is up to date
#!pip3 install --upgrade dsub

## Load in Variables

In [1]:
%run /home/jupyter/workspace/workspace-scripts/Var_set_2.ipynb

bucket_input:   gs://workspace-bucket-wb-blinding-cabbage-2295
bucket_output:  gs://workspace-output-wb-blinding-cabbage-2295
bucket_scripts: gs://script-wb-blinding-cabbage-2295
bucket_GenDat: gs://workspace-gendat-wb-blinding-cabbage-2295
dataset:        wb-silky-artichoke-2408.C2024Q3R8


In [4]:
# Save this Python variable as an environment variable so that its easier to use within %%bash cells
%env JOB_ID={LINE_COUNT_JOB_ID}
#my_bucket = os.environ['WORKSPACE_BUCKET']
pd.set_option('display.max_colwidth', 0)

results='grm' # Directory for results for dsu
%env results={results}

USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')
%env USER_NAME={USER_NAME}

JOB_NAME='clinvar_v8' # Job Name for dsu
%env JOB_NAME={JOB_NAME}

# ANCESTRY="EUR"
# %env ANCESTRY={ANCESTRY}

env: JOB_ID={LINE_COUNT_JOB_ID}
env: results=grm
env: USER_NAME='nathani
env: JOB_NAME=clinvar_v8


In [6]:
# Analysis Results Folder
output_folder = os.path.join(
    bucket_GenDat,
    results,
    JOB_NAME
)
%env output_folder={output_folder}

output_logs_folder = os.path.join(output_folder, "logs") # Log files folder
%env output_logs_folder={output_logs_folder}

env: output_folder=gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8
env: output_logs_folder=gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8/logs


# **PLINK format**

## Pruning

### make a script for pruning, MAF, Geno, hwe and autosome filtering

In [8]:
%%writefile /home/jupyter/workspace/workspace-scripts/dsub_Plink_auto.sh

set -o pipefail 
set -o errexit

plink2 \
    --bfile "${bed_file%.bed}" \
    --chr 1-22 \
    --maf 0.01 \
    --geno 0.1 \
    --hwe 1e-6 \
    --indep-pairwise 50 5 0.2 \
    --threads 7 \
    --memory 30000 \
    --make-bed \
    --out "${output_folder}/autosomes_pruned"

echo "output_folder: ${output_folder}"


Writing /home/jupyter/workspace/workspace-scripts/dsub_Plink_auto.sh


In [31]:
%%bash --out filtering

# Get a shorter username to leave more characters for the job name
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

MACHINE_TYPE="n2-standard-8"
#MACHINE_TYPE="n2-standard-16"
TIMESTAMP=$(date +"%Y%m%d_%H%M%S")
# Change for your bucket, path in output of cell directly above:
BASH_SCRIPT="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/dsub_Plink_auto.sh"

#google-cls-v2
    dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "gcr.io/bick-aps2/plink2:polygenic" \
    --network "global/networks/network" \
    --use-private-address \
    --subnetwork "regions/us-central1/subnetworks/subnetwork" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${output_logs_folder}/${TIMESTAMP}-{job-id}-{task-id}.log" \
    "$@" \
    --preemptible \
    --boot-disk-size 100 \
    --disk-size 200 \
    --machine-type ${MACHINE_TYPE} \
    --name "${JOB_NAME}" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --env output_folder="${output_folder}" \
    --input bed_file="gs://vwb-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bim_file="gs://vwb-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input fam_file="gs://vwb-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --output-recursive output_folder="${output_folder}"

Job properties:
  job-id: microarray--nathani--250908-032553-46
  job-name: microarray
  user-id: nathani
Provider internal-id (operation): projects/terra-vpc-sc-d55470c6/locations/us-central1/jobs/microarray--nathani--250908-032553-46-0-0
Launched job-id: microarray--nathani--250908-032553-46
To check the status, run:
  dstat --provider google-batch --project terra-vpc-sc-d55470c6 --location us-central1 --jobs 'microarray--nathani--250908-032553-46' --users 'nathani' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project terra-vpc-sc-d55470c6 --location us-central1 --jobs 'microarray--nathani--250908-032553-46' --users 'nathani'


In [5]:
#check job status
!dstat --provider google-batch --project terra-vpc-sc-d55470c6 --location us-central1 --jobs 'microarray--nathani--250908-032553-46' --users 'nathani' --status '*'

Job Name    Status                                      Last Update
----------  ------------------------------------------  --------------
microarray  Job state is set from RUNNING to SUCCEE...  09-08 06:43:42



In [11]:
#check log files
!gsutil ls {output_logs_folder}

gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250908_002502-microarray--nathani--250908-002504-65-task-None-stderr.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250908_002502-microarray--nathani--250908-002504-65-task-None-stdout.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250908_002502-microarray--nathani--250908-002504-65-task-None.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250908_032552-microarray--nathani--250908-032553-46-task-stderr.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250908_032552-microarray--nathani--250908-032553-46-task-stdout.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250908_032552-microarray--nathani--250908-032553-46-task.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/microarray-nathani-microarray--na

In [21]:
!gsutil cat gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.log

PLINK v2.00a2.3LM AVX2 Intel (24 Jan 2020)
Options in effect:
  --bfile /mnt/data/input/gs/fc-aou-datasets-controlled/v8/microarray/plink/arrays
  --chr 1-22
  --geno 0.1
  --hwe 1e-6
  --indep-pairwise 50 5 0.2
  --maf 0.01
  --make-bed
  --memory 60000
  --out /mnt/data/output/gs/fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned
  --threads 7

Hostname: 837c74752f15
Working directory: /mnt/data/workingdir
Start time: Mon Sep  8 04:39:24 2025

Random number seed: 1757306364
32101 MiB RAM detected; reserving 60000 MiB for main workspace.
Using up to 7 compute threads.
447278 samples (0 females, 0 males, 447278 ambiguous; 447278 founders) loaded
from
/mnt/data/input/gs/fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam.
1675355 out of 1739269 variants loaded from
/mnt/data/input/gs/fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim.
Note: No phenotype data present.
Calculating allele frequencies... done.
--geno: 466 variants removed due t

In [8]:
!gsutil ls {output_folder}

gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.bed
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.bim
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.fam
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.prune.in
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.prune.out
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/


In [9]:
!gsutil cat gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.bim | wc -l

102101


In [15]:
!gsutil cat gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.prune.out| wc -l 

27195


In [16]:
!gsutil cat gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.prune.in | wc -l 

74906


### extract pruned SNPs

In [48]:
%%writefile ~/dsub_Plink_extract.sh

set -o pipefail 
set -o errexit


plink2 \
    --bfile "${bed_file%.bed}" \
    --extract "${in_file}" \
    --threads 3 \
    --memory 12000 \
    --make-bed \
    --out "${output_folder}/autosomes_final_pruned"


Overwriting /home/jupyter/dsub_Plink_extract.sh


In [49]:
#move script to google cloud
!gsutil cp ~/dsub_Plink_extract.sh {my_bucket}/scripts/
!gsutil ls {my_bucket}/scripts/
 

Copying file:///home/jupyter/dsub_Plink_extract.sh [Content-Type=text/x-sh]...
/ [1 files][  214.0 B/  214.0 B]                                                
Operation completed over 1 objects/214.0 B.                                      
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/dsub_PLINK2_merge_bgen_files_for_GRM_Bgen.v8.sh
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/dsub_Plink_auto.sh
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/dsub_Plink_extract.sh
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/dsub_Plink_pruning.sh
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/dsub_bgenix_indexing_bgen_files_for_GRM_Bgen.v8.sh
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/dsub_create_smaller_PLINK_bgen_files_for_GRM_Bgen.v8.sh


In [58]:
%%bash --out extracting

# Get a shorter username to leave more characters for the job name
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

MACHINE_TYPE="n2-standard-4"
#MACHINE_TYPE="n2-standard-16"
TIMESTAMP=$(date +"%Y%m%d_%H%M%S")
# Change for your bucket, path in output of cell directly above:
BASH_SCRIPT="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/dsub_Plink_extract.sh"

#clear log folder
gsutil -m rm "${output_logs_folder}/**"

#google-cls-v2
    dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "gcr.io/bick-aps2/plink2:polygenic" \
    --network "global/networks/network" \
    --use-private-address \
    --subnetwork "regions/us-central1/subnetworks/subnetwork" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${output_logs_folder}/${TIMESTAMP}-{job-id}-{task-id}.log" \
    "$@" \
    --preemptible \
    --boot-disk-size 100 \
    --disk-size 400 \
    --machine-type ${MACHINE_TYPE} \
    --name "${JOB_NAME}" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --env output_folder="${output_folder}" \
    --input bed_file="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.bed" \
    --input bim_file="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.bim" \
    --input fam_file="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.fam" \
    --input in_file="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.prune.in" \
    --output-recursive output_folder="${output_folder}"

Removing gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250909_034037-microarray--nathani--250909-034040-27-task-stderr.log...
Removing gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250909_034037-microarray--nathani--250909-034040-27-task-stdout.log...
Removing gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250909_034037-microarray--nathani--250909-034040-27-task.log...
/ [3/3 objects] 100% Done                                                       
Operation completed over 3 objects.                                              
Job properties:
  job-id: microarray--nathani--250909-035021-81
  job-name: microarray
  user-id: nathani
Provider internal-id (operation): projects/terra-vpc-sc-d55470c6/locations/us-central1/jobs/microarray--nathani--250909-035021-81-0-0
Launched job-id: microarray--nathani--250909-035021-81
To check the status, run:
  dstat --provider google-batch --project t

In [59]:
%env JOB_ID={extracting}

env: JOB_ID=microarray--nathani--250909-035021-81


In [62]:
%%bash

dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

Job Name    Status                                      Last Update
----------  ------------------------------------------  --------------
microarray  Job state is set from RUNNING to SUCCEE...  09-09 03:57:56



In [63]:
!gsutil ls {output_logs_folder}

gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250909_035018-microarray--nathani--250909-035021-81-task-stderr.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250909_035018-microarray--nathani--250909-035021-81-task-stdout.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250909_035018-microarray--nathani--250909-035021-81-task.log


In [68]:
!gsutil ls {output_folder}

gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.bed
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.bim
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.fam
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.bed
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.bim
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.fam
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.prune.in
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.prune.out
gs://fc-secure-0c09c643-6

In [69]:
!gsutil cat gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.bim | wc -l
!gsutil cat gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.fam | wc -l

74906
447278


In [71]:
!gsutil cat gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.fam | cut -f1 | sort | uniq -c

 447278 0


In [ ]:
!gsutil -{PROJECT_ID} 